# Part 3: Sentiment Model

<div dir="rtl">

این نوت‌بوک اولین نوت‌بوک بخش سوم پروژه است: پیش‌بینی امتیاز ۱ تا ۵ از متن نظر. داده پاک می‌شود، بررسی می‌شود که test چه توزیعی از امتیازها دارد، validation و holdout متوازن جدا می‌شوند و مدل‌های TF-IDF با هم مقایسه می‌شوند. fine-tune کردن ترنسفورمرها در part3_finetune.ipynb است، و انتخاب مدل نهایی، ترکیب مدل‌ها، ارزیابی روی holdout و فایل submission در part3_final.ipynb.

فایل‌های لازم در پوشه‌ی project2 در Google Drive: train_data.csv و test_data.csv. خروجی‌ها در همان پوشه ذخیره می‌شوند: part3_train.parquet و part3_test.parquet (داده‌ی پاک‌شده همراه تقسیم داده)، part3_tfidf.joblib (مدل TF-IDF) و part3_tfidf_probs.parquet (احتمال‌های این مدل برای validation، holdout و test).

</div>

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = 'plotly_white'
path = '/content/drive/MyDrive/project2/'

## Data

<div dir="rtl">

مثل بخش ۱، نظرهای تکراری (همان نویسنده، همان محصول، همان متن) حذف شدند. از ستون‌های test فقط متن نظر و summary به کار می‌آیند.

</div>

In [3]:
cols = ['overall', 'reviewerID', 'asin', 'reviewText', 'summary']
train = pd.read_csv(path + 'train_data.csv', usecols=cols)
test = pd.read_csv(path + 'test_data.csv')
print('train:', train.shape, ' test:', test.shape)

train = train.drop_duplicates(['reviewerID', 'asin', 'reviewText'])
print('after removing duplicate reviews:', len(train))

train: (838944, 5)  test: (20000, 10)
after removing duplicate reviews: 830390


### Cleaning

<div dir="rtl">

اینجا فقط کارهایی انجام شد که برای همه‌ی مدل‌ها لازم است: حذف کد HTML (ویدئو و لینک محصول)، حذف آدرس‌های اینترنتی و یکی کردن فاصله‌ها. حروف کوچک نشدند و stop wordها حذف نشدند. ترنسفورمرها روی متن کامل و با حروف بزرگ آموزش دیده‌اند و کلماتی مثل not معنی جمله را برعکس می‌کنند. هر مدل پیش‌پردازش خاص خودش را در بخش خودش دارد.

</div>

In [4]:
def clean(text):
    text = text.fillna('').str.replace(r'<[^>]+>', ' ', regex=True)
    text = text.str.replace(r'http\S+|www\.\S+', ' ', regex=True)
    return text.str.replace(r'\s+', ' ', regex=True).str.strip()

for df in [train, test]:
    df['reviewText'] = clean(df['reviewText'])
    df['summary'] = clean(df['summary'])

print('empty reviews in train:', (train['reviewText'] == '').sum(), ' in test:', (test['reviewText'] == '').sum())
train = train[train['reviewText'] != '']

empty reviews in train: 12  in test: 0


### Repeated texts

<div dir="rtl">

بعضی نویسنده‌ها یک متن را برای چند محصول گذاشته‌اند. این نسخه‌ها اطلاعات تازه‌ای به مدل نمی‌دهند و اگر یکی در داده‌ی آموزش و دیگری در validation بیفتد، مدلی که متن‌ها را حفظ می‌کند بهتر از واقعیت دیده می‌شود. از هر متن فقط یک نسخه نگه داشته شد.

</div>

In [5]:
norm = train['reviewText'].str.lower()
repeated = norm.duplicated(keep=False)
print('reviews whose text appears more than once:', repeated.sum())
print('repeated texts with a single rating:', round((train[repeated].groupby(norm[repeated])['overall'].nunique() == 1).mean(), 3))

train = train[~norm.duplicated()]
print('after keeping one copy of each text:', len(train))
del norm, repeated

reviews whose text appears more than once: 42290
repeated texts with a single rating: 0.993
after keeping one copy of each text: 807413


In [6]:
seen = test['reviewText'].str.lower().isin(set(train['reviewText'].str.lower()))
print('test reviews whose text is also in train:', seen.sum(), f'({seen.mean():.1%})')

test reviews whose text is also in train: 1267 (6.3%)


<div dir="rtl">

این نظرهای test در train هم هستند و امتیازشان آنجا معلوم است. می‌شد امتیاز را از train کپی کرد، ولی این کار مدل‌سازی نیست و از تکرار داده استفاده می‌کند؛ پس کنار گذاشته شد و مدل همه‌ی test را خودش پیش‌بینی می‌کند.

</div>

<div dir="rtl">

نسخه‌های تقریباً یکسان هم هستند: همان نویسنده و همان محصول با متن کمی ویرایش‌شده، یا همان نویسنده با ۱۵۰ کاراکتر اول یکسان. این‌ها در داده‌ی آموزش می‌مانند ولی برای validation و holdout انتخاب نمی‌شوند.

</div>

In [7]:
head = train['reviewText'].str.lower().str[:150]
near = train.assign(head=head).duplicated(['reviewerID', 'head'], keep=False) | train.duplicated(['reviewerID', 'asin'], keep=False)
print('near-duplicate reviews:', near.sum())
del head

near-duplicate reviews: 3916


## Is the test set like the training data?

<div dir="rtl">

test برچسب ندارد، پس توزیع امتیازهایش مستقیم دیده نمی‌شود. سه نشانه‌ی غیرمستقیم بررسی شد: کلمات مثبت و منفی، پیش‌بینی یک مدل ساده، و برآورد سهم هر امتیاز با دو روش آماری. در این بخش فقط متن test استفاده می‌شود و هیچ تصمیمی روی عددهای test تنظیم نمی‌شود.

</div>

In [8]:
def text_of(df):
    return (df['summary'] + ' ' + df['reviewText']).str.lower()

positive_words = r'\b(?:great|love|perfect|excellent|awesome)\b'
negative_words = r'\b(?:return|broke|stopped|waste|terrible|junk|disappoint)'
pd.DataFrame({'train': [text_of(train).str.contains(w).mean() for w in [positive_words, negative_words]],
              'test': [text_of(test).str.contains(w).mean() for w in [positive_words, negative_words]]},
             index=['positive words', 'negative words']).round(3)

,train,test
positive words,0.507,0.370
negative words,0.158,0.252


<div dir="rtl">

برای نشانه‌ی دوم یک مدل ساده (TF-IDF و logistic regression روی ۱۰۰ هزار نظر) فقط برای همین بررسی آموزش دید. مدل‌های اصلی در بخش‌های بعد ساخته می‌شوند.

</div>

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

probe_fit = train.sample(100000, random_state=0)
probe_held = train.drop(probe_fit.index).sample(30000, random_state=1)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200000, sublinear_tf=True)
probe = LogisticRegression(C=4, solver='saga', tol=1e-3, max_iter=100, random_state=0)
probe.fit(vectorizer.fit_transform(text_of(probe_fit)), probe_fit['overall'])

held_prob = probe.predict_proba(vectorizer.transform(text_of(probe_held)))
test_prob = probe.predict_proba(vectorizer.transform(text_of(test)))
print('accuracy on held-out train reviews:', round((held_prob.argmax(1) + 1 == probe_held['overall']).mean(), 3))

accuracy on held-out train reviews: 0.7


In [10]:
def shares(labels):
    return pd.Series(labels).value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0).values

pd.DataFrame({'train, true': shares(probe_held['overall']),
              'train, predicted': shares(held_prob.argmax(1) + 1),
              'test, predicted': shares(test_prob.argmax(1) + 1)}, index=[1, 2, 3, 4, 5]).round(3)

,"train, true","train, predicted","test, predicted"
1,0.099,0.104,0.225
2,0.066,0.034,0.081
3,0.096,0.073,0.149
4,0.187,0.152,0.180
5,0.551,0.636,0.366


<div dir="rtl">

نشانه‌ی سوم دو روش استاندارد برای برآورد سهم کلاس‌ها در داده‌ی بدون برچسب است:

- EM (Saerens و همکاران، ۲۰۰۲): احتمال‌های مدل برای هر نظر با سهم‌های فرضی کلاس‌ها اصلاح می‌شوند و سهم‌ها آن‌قدر به‌روز می‌شوند که با میانگین احتمال‌های اصلاح‌شده جور شوند.
- BBSE (Lipton و همکاران، ۲۰۱۸): ماتریس درهم‌ریختگی مدل روی نظرهای برچسب‌دار نشان می‌دهد هر امتیاز واقعی با چه سهمی به هر امتیاز پیش‌بینی می‌شود. با حل یک دستگاه معادله، از سهم پیش‌بینی‌ها در test به سهم امتیازهای واقعی می‌رسیم.

برای اینکه خطای خود روش‌ها معلوم شود، همین برآوردها روی یک نمونه‌ی واقعاً متوازن از train (۲۰۰۰ نظر از هر امتیاز) هم حساب شد.

</div>

In [11]:
from sklearn.metrics import confusion_matrix

prior = probe_fit['overall'].value_counts(normalize=True).sort_index().values

def em_estimate(prob, steps=200):
    q = prior.copy()
    for _ in range(steps):
        w = prob * q / prior
        w = w / w.sum(axis=1, keepdims=True)
        q = w.mean(axis=0)
    return q

def bbse_estimate(prob):
    joint = confusion_matrix(probe_held['overall'], held_prob.argmax(1) + 1, normalize='all')
    w = np.linalg.solve(joint.T, shares(prob.argmax(1) + 1))
    q = w * joint.sum(axis=1)
    return q / q.sum()

check_sample = train.drop(probe_fit.index).groupby('overall').sample(2000, random_state=2)
check_prob = probe.predict_proba(vectorizer.transform(text_of(check_sample)))

estimates = pd.DataFrame({'train': prior,
                          'test, EM': em_estimate(test_prob),
                          'test, BBSE': bbse_estimate(test_prob),
                          'balanced check, EM': em_estimate(check_prob),
                          'balanced check, BBSE': bbse_estimate(check_prob)}, index=[1, 2, 3, 4, 5])
estimates.round(3)

,train,"test, EM","test, BBSE","balanced check, EM","balanced check, BBSE"
1,0.099,0.223,0.200,0.221,0.190
2,0.066,0.162,0.203,0.170,0.229
3,0.097,0.230,0.214,0.224,0.191
4,0.187,0.176,0.186,0.172,0.183
5,0.551,0.208,0.197,0.213,0.206


In [12]:
plot = estimates[['train', 'test, EM', 'test, BBSE']].reset_index(names='rating').melt(id_vars='rating', var_name='data', value_name='share')
fig = px.bar(plot, x='rating', y='share', color='data', barmode='group', text_auto='.0%',
             title='Share of each rating: train vs. estimates for the test set')
fig.update_layout(xaxis_type='category', yaxis_tickformat='.0%')
fig.show()

<div dir="rtl">

اگر test متوازن باشد، مدلی که از توزیع train یاد گرفته زیادی ۵ پیش‌بینی می‌کند. ساده‌ترین اصلاح این است که احتمال هر امتیاز بر سهمش در train تقسیم شود؛ یعنی «عادت» مدل به گفتن ۵ کنار برود. اثرش روی همان نمونه‌ی متوازن:

</div>

In [13]:
plain = check_prob.argmax(1) + 1
corrected = (check_prob / prior).argmax(1) + 1
print('balanced sample  plain:', round((plain == check_sample['overall']).mean(), 3),
      ' with prior correction:', round((corrected == check_sample['overall']).mean(), 3))

plain_held = held_prob.argmax(1) + 1
corrected_held = (held_prob / prior).argmax(1) + 1
print('train mix        plain:', round((plain_held == probe_held['overall']).mean(), 3),
      ' with prior correction:', round((corrected_held == probe_held['overall']).mean(), 3))

balanced sample  plain: 0.508  with prior correction: 0.544
train mix        plain: 0.7  with prior correction: 0.654


<div dir="rtl">

در test کلمات مثبت کمتر آمده‌اند (۳۷٪ نظرها در برابر ۵۱٪ در train) و کلمات منفی بیشتر (۲۵٪ در برابر ۱۶٪). مدل ساده روی نظرهای train ۶۴٪ را ۵ پیش‌بینی می‌کند و روی test فقط ۳۷٪. هر دو روش آماری سهم هر امتیاز در test را بین ۱۶٪ و ۲۳٪ برآورد می‌کنند، یعنی حدود ۲۰٪، و روی نمونه‌ی متوازن train هم همین اندازه خطا دارند. پس test به احتمال زیاد از هر امتیاز تعداد برابری دارد، در حالی که در train امتیاز ۵ بیش از نیمی از نظرهاست.

این نتیجه دو پیامد دارد. اول، validation باید متوازن باشد: اصلاح prior درستی مدل ساده را روی نمونه‌ی متوازن از ۰.۵۰۹ به ۰.۵۴۶ می‌رساند ولی روی توزیع train از ۰.۷۰ به ۰.۶۵ پایین می‌آورد، پس validation با توزیع train همین تغییر مفید را رد می‌کرد. دوم، همیشه ۵ گفتن روی test حدود ۲۰٪ درستی دارد، نه ۵۵٪. در بخش‌های بعد سه راه برای این تفاوت مقایسه می‌شود: آموزش روی داده‌ی متوازن، وزن دادن به کلاس‌ها و اصلاح prior. عددهای برآوردشده برای test در هیچ مدلی استفاده نمی‌شوند و هدف سهم برابر برای همه‌ی امتیازهاست.

</div>

## Validation and holdout

<div dir="rtl">

از هر امتیاز ۲۰۰۰ نظر برای validation جدا شد که مدل‌ها و تنظیم‌ها با آن انتخاب می‌شوند، و ۲۰۰۰ نظر دیگر برای holdout که فقط یک بار در آخر برای گزارش مدل نهایی استفاده می‌شود. هر دو متوازن‌اند تا شبیه test باشند، و فقط از نظرهایی انتخاب شدند که نسخه‌ی تقریباً یکسانی ندارند. بقیه‌ی نظرها داده‌ی آموزش‌اند و هر مدل نمونه‌ی آموزشش را از آن‌ها برمی‌دارد.

</div>

In [14]:
candidates = train[~near]
valid = candidates.groupby('overall').sample(2000, random_state=0)
holdout = candidates.drop(valid.index).groupby('overall').sample(2000, random_state=1)

train['set'] = 'train'
train.loc[valid.index, 'set'] = 'valid'
train.loc[holdout.index, 'set'] = 'holdout'
pd.crosstab(train['overall'], train['set'], margins=True)

set,holdout,train,valid,All
overall,,,,
1,2000,75512,2000,79512
2,2000,50729,2000,54729
3,2000,74287,2000,78287
4,2000,147145,2000,151145
5,2000,439740,2000,443740
All,10000,787413,10000,807413


<div dir="rtl">

بررسی نشتی: هیچ متن و هیچ جفت نویسنده و محصولی از validation و holdout در داده‌ی آموزش نیست.

</div>

In [15]:
fit_part = train[train['set'] == 'train']
fit_texts = set(fit_part['reviewText'].str.lower())
fit_pairs = set(zip(fit_part['reviewerID'], fit_part['asin']))
for s in ['valid', 'holdout']:
    part = train[train['set'] == s]
    print(s, ' same text in train:', part['reviewText'].str.lower().isin(fit_texts).sum(),
          ' same reviewer and product in train:', sum(p in fit_pairs for p in zip(part['reviewerID'], part['asin'])))
del fit_part, fit_texts, fit_pairs

valid  same text in train: 0  same reviewer and product in train: 0
holdout  same text in train: 0  same reviewer and product in train: 0


<div dir="rtl">

داده‌ی پاک‌شده همراه ستون set ذخیره می‌شود تا نوت‌بوک fine-tune دقیقاً همین تقسیم را بخواند.

</div>

In [16]:
train[['overall', 'reviewerID', 'asin', 'summary', 'reviewText', 'set']].to_parquet(path + 'part3_train.parquet')
test[['summary', 'reviewText']].to_parquet(path + 'part3_test.parquet')

## TF-IDF models

<div dir="rtl">

اولین خانواده‌ی مدل‌ها: هر نظر به برداری از وزن کلمات و جفت‌کلمه‌ها تبدیل می‌شود و یک مدل خطی روی آن آموزش می‌بیند. وزن هر کلمه TF-IDF است: هرچه کلمه در یک نظر بیشتر و در کل نظرها کمتر آمده باشد، وزنش بیشتر است. کلمه‌ای مثل the که در همه‌ی نظرها هست وزن نزدیک صفر می‌گیرد، پس حذف دستی stop wordها لازم نیست. جفت‌کلمه‌ها (bigram) عبارت‌هایی مثل not good و not bad را از good و bad جدا می‌کنند.

این مدل‌ها سریع‌اند، اثر هر کلمه را نشان می‌دهند و مبنای مقایسه با ترنسفورمرها هستند. همه‌ی مقایسه‌ها روی validation متوازن است. برای سرعت، تنظیم‌ها روی ۱۰۰ هزار نظر انتخاب شدند و اثر داده‌ی بیشتر در آخر جدا بررسی شد. TF-IDF هر بار فقط روی نمونه‌ی آموزش fit می‌شود.

</div>

In [17]:
import gc

del probe, vectorizer, held_prob, test_prob, check_prob
gc.collect()

from scipy.sparse import hstack, diags
from sklearn.metrics import accuracy_score, f1_score

fit_pool = train[train['set'] == 'train']
valid = train[train['set'] == 'valid']
y_valid = valid['overall'].values
results = []

def score(stage, name, pred):
    results.append({'stage': stage, 'model': name, 'accuracy': accuracy_score(y_valid, pred),
                    'macro_f1': f1_score(y_valid, pred, average='macro')})
    print(f'{name:50s} accuracy {results[-1]["accuracy"]:.4f}   macro-F1 {results[-1]["macro_f1"]:.4f}')

def joined(df):
    return df['summary'] + ' . ' + df['reviewText']

token_pattern = r"(?u)\b\w[\w']*\b"

def word_tfidf(**options):
    settings = dict(ngram_range=(1, 2), min_df=3, max_features=300000, sublinear_tf=True,
                    token_pattern=token_pattern, dtype=np.float32)
    settings.update(options)
    return TfidfVectorizer(**settings)

def lr(C, **options):
    return LogisticRegression(C=C, solver='saga', tol=1e-3, max_iter=300, random_state=0, **options)

<div dir="rtl">

الگوی token طوری نوشته شد که don't و isn't یک کلمه بمانند و عددها و کلمه‌های تک‌حرفی هم حذف نشوند؛ الگوی پیش‌فرض scikit-learn don't را به don تبدیل می‌کند و 5 را در «5 stars» دور می‌ریزد. sublinear_tf به‌جای تعداد تکرار از لگاریتم آن استفاده می‌کند تا نظرهای بلند با تکرار یک کلمه وزن بی‌جا نگیرند.

</div>

### Balancing

<div dir="rtl">

سه راه برای اینکه مدل به امتیاز ۵ عادت نکند، با اندازه‌ی یکسان ۱۰۰ هزار نظر مقایسه شد: نمونه‌ی متوازن (۲۰ هزار از هر امتیاز)، نمونه‌ی طبیعی با class_weight (خطای امتیازهای کم‌تعداد وزن بیشتری می‌گیرد) و نمونه‌ی طبیعی با اصلاح prior. نمونه‌ی طبیعی بدون اصلاح هم برای مقایسه آمده. مقدار C (عکس قدرت regularization) برای هر روش جدا انتخاب شد.

</div>

In [18]:
natural = fit_pool.sample(100000, random_state=0)
balanced = fit_pool.groupby('overall').sample(20000, random_state=0)
natural_prior = natural['overall'].value_counts(normalize=True).sort_index().values
print('rating 2 reviews  natural sample:', (natural['overall'] == 2).sum(), ' balanced sample:', (balanced['overall'] == 2).sum())

vectorizer = word_tfidf()
X = vectorizer.fit_transform(joined(natural))
X_valid = vectorizer.transform(joined(valid))
for C in [0.25, 1, 4]:
    prob = lr(C).fit(X, natural['overall']).predict_proba(X_valid)
    score('balancing', f'natural, C={C}', prob.argmax(1) + 1)
    score('balancing', f'natural + prior correction, C={C}', (prob / natural_prior).argmax(1) + 1)
    score('balancing', f'natural + class_weight, C={C}', lr(C, class_weight='balanced').fit(X, natural['overall']).predict(X_valid))

rating 2 reviews  natural sample: 6509  balanced sample: 20000
natural, C=0.25                                    accuracy 0.4182   macro-F1 0.3549
natural + prior correction, C=0.25                 accuracy 0.5641   macro-F1 0.5574
natural + class_weight, C=0.25                     accuracy 0.5625   macro-F1 0.5531
natural, C=1                                       accuracy 0.4959   macro-F1 0.4572
natural + prior correction, C=1                    accuracy 0.5731   macro-F1 0.5689


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning:

The max_iter was reached which means the coef_ did not converge



natural + class_weight, C=1                        accuracy 0.5681   macro-F1 0.5584
natural, C=4                                       accuracy 0.5156   macro-F1 0.4855
natural + prior correction, C=4                    accuracy 0.5693   macro-F1 0.5637
natural + class_weight, C=4                        accuracy 0.5541   macro-F1 0.5389


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning:

The max_iter was reached which means the coef_ did not converge



In [19]:
vectorizer = word_tfidf()
X = vectorizer.fit_transform(joined(balanced))
X_valid = vectorizer.transform(joined(valid))
balanced_accuracy = {}
for C in [0.25, 1, 4]:
    pred = lr(C).fit(X, balanced['overall']).predict(X_valid)
    score('balancing', f'balanced, C={C}', pred)
    balanced_accuracy[C] = accuracy_score(y_valid, pred)

best_C = max(balanced_accuracy, key=balanced_accuracy.get)
print('C for the next steps:', best_C)

balanced, C=0.25                                   accuracy 0.5803   macro-F1 0.5745
balanced, C=1                                      accuracy 0.5845   macro-F1 0.5807
balanced, C=4                                      accuracy 0.5746   macro-F1 0.5718
C for the next steps: 1


<div dir="rtl">

نمونه‌ی متوازن با C=1 بهترین نتیجه را دارد (۰.۵۸۵). روی نمونه‌ی طبیعی، اصلاح prior (۰.۵۷۳) و class_weight (۰.۵۶۴) خیلی کمک می‌کنند ولی به نمونه‌ی متوازن نمی‌رسند، و مدل بدون اصلاح با ۰.۴۹۶ خیلی عقب است. دلیلش تعداد مثال‌هاست: نمونه‌ی طبیعی فقط ۶٬۵۰۹ نظر با امتیاز ۲ دارد و نمونه‌ی متوازن ۲۰ هزار. وزن دادن یا اصلاح احتمال‌ها مثال تازه‌ای به مدل نمی‌دهد. الگوریتم saga برای class_weight با C=1 و C=4 در ۳۰۰ دور کامل همگرا نشد (هشدارهای بالا)، ولی فاصله‌ی این دو ردیف با نمونه‌ی متوازن ۲ تا ۳ امتیاز است. در هر سه روش C=1 بهترین است. از اینجا به بعد نمونه‌ی متوازن و C=1 استفاده شد.

</div>

### Text features

<div dir="rtl">

با نمونه‌ی متوازن و C انتخاب‌شده، هر بار یک چیز در ساختن ویژگی‌ها عوض شد تا اثرش جدا دیده شود. ردیف پایه همان مدل بخش قبل است: summary و متن نظر پشت هم، کلمات و جفت‌کلمه‌ها.

</div>

In [20]:
y_fit = balanced['overall']

def feature_test(name, fit_text, valid_text, **options):
    vectorizer = word_tfidf(**options)
    X = vectorizer.fit_transform(fit_text)
    score('features', name, lr(best_C).fit(X, y_fit).predict(vectorizer.transform(valid_text)))

feature_test('review text only', balanced['reviewText'], valid['reviewText'])
feature_test('summary + review (base)', joined(balanced), joined(valid))
feature_test('words only, no bigrams', joined(balanced), joined(valid), ngram_range=(1, 1))
feature_test('default scikit-learn tokens', joined(balanced), joined(valid), token_pattern=r'(?u)\b\w\w+\b')
feature_test('stop words removed', joined(balanced), joined(valid), stop_words='english')

review text only                                   accuracy 0.5611   macro-F1 0.5566
summary + review (base)                            accuracy 0.5845   macro-F1 0.5807
words only, no bigrams                             accuracy 0.5479   macro-F1 0.5438
default scikit-learn tokens                        accuracy 0.5766   macro-F1 0.5726
stop words removed                                 accuracy 0.5399   macro-F1 0.5341


<div dir="rtl">

فهرست stop wordهای scikit-learn کلماتی مثل not، no، never، nothing و cannot را هم دارد:

</div>

In [21]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
print(sorted(ENGLISH_STOP_WORDS & {'not', 'no', 'never', 'nothing', 'cannot', 'nor', 'without', 'but', 'very', 'too', 'few', 'less'}))

['but', 'cannot', 'few', 'less', 'never', 'no', 'nor', 'not', 'nothing', 'too', 'very', 'without']


<div dir="rtl">

ریشه‌یابی (stemming) شکل‌های یک کلمه را یکی می‌کند (worked، working و works همه work می‌شوند). Snowball stemmer از کتابخانه‌ی nltk استفاده شد.

</div>

In [22]:
import re
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer('english')
stems = {}

def stemmed(text):
    words = re.findall(token_pattern, text.lower())
    return ' '.join(stems.setdefault(w, stemmer.stem(w)) for w in words)

print(stemmed("Worked great at first, but it stopped working and the batteries don't charge"))
feature_test('stemmed', joined(balanced).map(stemmed), joined(valid).map(stemmed))

work great at first but it stop work and the batteri don't charg
stemmed                                            accuracy 0.5850   macro-F1 0.5811


<div dir="rtl">

دو تغییر دیگر ویژگی اضافه می‌کنند: summary با ویژگی‌های جدا (great در summary وزنی جدا از great در متن می‌گیرد) و n-gramهای حرفی (تکه‌های ۲ تا ۵ حرفی داخل هر کلمه) که غلط‌های املایی، کلمه‌های چسبیده و علامت‌هایی مثل !!! را هم می‌گیرند.

</div>

In [23]:
def text_part(df, name):
    return {'summary': df['summary'], 'review': df['reviewText']}.get(name, joined(df))

def new_vectorizer(name):
    if name == 'summary':
        return word_tfidf(max_features=50000)
    if name == 'char':
        return TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=5, max_features=300000,
                               sublinear_tf=True, dtype=np.float32)
    return word_tfidf()

def fit_features(df, names):
    vectorizers = {name: new_vectorizer(name) for name in names}
    X = hstack([vectorizers[n].fit_transform(text_part(df, n)) for n in names]).tocsr()
    return vectorizers, X

def transform(vectorizers, df):
    return hstack([v.transform(text_part(df, n)) for n, v in vectorizers.items()]).tocsr()

for name, names in [('summary and review as separate features', ['summary', 'review']),
                    ('base + character n-grams', ['words', 'char'])]:
    vectorizers, X = fit_features(balanced, names)
    score('features', name, lr(best_C).fit(X, y_fit).predict(transform(vectorizers, valid)))
    del vectorizers, X
    gc.collect()

summary and review as separate features            accuracy 0.5826   macro-F1 0.5798
base + character n-grams                           accuracy 0.5787   macro-F1 0.5755


<div dir="rtl">

summary مهم است: بدون آن درستی ۲.۴ امتیاز پایین می‌آید (۰.۵۶۱ در برابر ۰.۵۸۵). حذف جفت‌کلمه‌ها ۳.۷ امتیاز و حذف stop wordها ۴.۵ امتیاز درستی را کم می‌کند، چون فهرست stop wordهای scikit-learn کلماتی مثل not، no، never و but را هم دارد. الگوی پیش‌فرض token ۰.۸ امتیاز بدتر است. stemming (۰.۵۸۵)، ویژگی‌های جدا برای summary (۰.۵۸۳) و n-gramهای حرفی (۰.۵۷۹) سودی نداشتند. پس ویژگی‌های پایه بدون تغییر ماندند: summary و متن نظر پشت هم، کلمات و جفت‌کلمه‌ها، بدون حذف stop word و بدون stemming.

</div>

### Models

<div dir="rtl">

با همین ویژگی‌ها چند خانواده‌ی مدل مقایسه شدند و برای هر کدام تنظیم اصلی‌اش با چند مقدار امتحان شد.

</div>

In [24]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB

vectorizer = word_tfidf()
X = vectorizer.fit_transform(joined(balanced))
X_valid = vectorizer.transform(joined(valid))

score('models', f'logistic regression, C={best_C}', lr(best_C).fit(X, y_fit).predict(X_valid))
for C in [0.03, 0.1, 0.3]:
    score('models', f'linear SVM, C={C}', LinearSVC(C=C).fit(X, y_fit).predict(X_valid))
for alpha in [3, 10, 30]:
    score('models', f'complement naive Bayes, alpha={alpha}', ComplementNB(alpha=alpha).fit(X, y_fit).predict(X_valid))

logistic regression, C=1                           accuracy 0.5845   macro-F1 0.5807
linear SVM, C=0.03                                 accuracy 0.5686   macro-F1 0.5542
linear SVM, C=0.1                                  accuracy 0.5801   macro-F1 0.5696
linear SVM, C=0.3                                  accuracy 0.5743   macro-F1 0.5660
complement naive Bayes, alpha=3                    accuracy 0.5346   macro-F1 0.5105
complement naive Bayes, alpha=10                   accuracy 0.5366   macro-F1 0.5172
complement naive Bayes, alpha=30                   accuracy 0.5297   macro-F1 0.5133


<div dir="rtl">

NB-SVM (Wang و Manning، ۲۰۱۲) قبل از مدل خطی، هر ویژگی را در نسبت Naive Bayes آن ضرب می‌کند: لگاریتم اینکه آن ویژگی در یک امتیاز چقدر بیشتر از بقیه‌ی امتیازها آمده است. برای هر امتیاز یک مدل «این امتیاز یا بقیه» آموزش دید و امتیازی که بیشترین خروجی را دارد انتخاب شد.

</div>

In [25]:
def nb_ratio(X, is_class, alpha=1):
    present = (X > 0).astype(np.float32)
    p = alpha + np.asarray(present[is_class].sum(axis=0)).ravel()
    q = alpha + np.asarray(present[~is_class].sum(axis=0)).ravel()
    return np.log((p / p.sum()) / (q / q.sum()))

for C in [1, 4, 16]:
    outputs = []
    for rating in [1, 2, 3, 4, 5]:
        is_class = (y_fit == rating).values
        r = diags(nb_ratio(X, is_class))
        model = lr(C).fit(X @ r, is_class)
        outputs.append(model.decision_function(X_valid @ r))
    score('models', f'NB-SVM, C={C}', np.argmax(outputs, axis=0) + 1)

NB-SVM, C=1                                        accuracy 0.5783   macro-F1 0.5674
NB-SVM, C=4                                        accuracy 0.5814   macro-F1 0.5737
NB-SVM, C=16                                       accuracy 0.5639   macro-F1 0.5583


<div dir="rtl">

امتیازها ترتیب دارند: ۴ به ۵ نزدیک‌تر است تا به ۱. Ridge regression امتیاز را یک عدد پیوسته پیش‌بینی می‌کند و دو راه برای تبدیل آن به ۱ تا ۵ امتحان شد: گرد کردن، و مرزهایی که سهم هر امتیاز را ۲۰٪ کنند، چون هدف سهم برابر است. مرزها از پیش‌بینی‌های cross-validation روی خود نمونه‌ی آموزش به دست آمدند، نه از validation.

</div>

In [26]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict

for alpha in [1, 3]:
    ridge = Ridge(alpha=alpha)
    out_of_fold = cross_val_predict(ridge, X, y_fit, cv=5)
    cuts = np.quantile(out_of_fold, [0.2, 0.4, 0.6, 0.8])
    valid_value = ridge.fit(X, y_fit).predict(X_valid)
    score('models', f'ridge regression rounded, alpha={alpha}', np.clip(np.round(valid_value), 1, 5))
    score('models', f'ridge regression equal-share cuts, alpha={alpha}', np.digitize(valid_value, cuts) + 1)

ridge regression rounded, alpha=1                  accuracy 0.4612   macro-F1 0.4625
ridge regression equal-share cuts, alpha=1         accuracy 0.4715   macro-F1 0.4401
ridge regression rounded, alpha=3                  accuracy 0.4494   macro-F1 0.4456
ridge regression equal-share cuts, alpha=3         accuracy 0.4755   macro-F1 0.4401


<div dir="rtl">

مدل درختی هم مقایسه شد: LightGBM (gradient boosting). روی ۳۰۰ هزار ویژگی تنک بیش از حد کند بود، پس به LightGBM و به logistic regression هر دو فقط همان ۵۰۰۰ کلمه و جفت‌کلمه‌ی پرتکرار داده شد تا مقایسه منصفانه باشد.

</div>

In [27]:
import lightgbm as lgb

small_vectorizer = word_tfidf(max_features=5000)
X_small = small_vectorizer.fit_transform(joined(balanced))
X_small_valid = small_vectorizer.transform(joined(valid))

import time

gbm = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.1, num_leaves=31, colsample_bytree=0.3,
                         random_state=0, verbose=-1)
for name, model in [('LightGBM, 5k features', gbm), (f'logistic regression, 5k features, C={best_C}', lr(best_C))]:
    start = time.time()
    model.fit(X_small, y_fit)
    seconds = time.time() - start
    score('models', name, model.predict(X_small_valid))
    print('  training time:', round(seconds), 'seconds')

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



LightGBM, 5k features                              accuracy 0.5712   macro-F1 0.5670
  training time: 591 seconds
logistic regression, 5k features, C=1              accuracy 0.5710   macro-F1 0.5673
  training time: 11 seconds


In [28]:
models = pd.DataFrame([r for r in results if r['stage'] == 'models'])
models['family'] = models['model'].str.split(',').str[0]
models.loc[models['model'].str.contains('5k'), 'family'] += ' (5k features)'
best = models.sort_values('accuracy').groupby('family').tail(1).sort_values('accuracy')

fig = px.bar(best, x='accuracy', y='family', orientation='h', text_auto='.3f', hover_data=['model', 'macro_f1'],
             title='Best setting of each model family on balanced validation')
fig.update_xaxes(range=[0.4, 0.62])
fig.show()

<div dir="rtl">

logistic regression با ۰.۵۸۵ بهترین است. NB-SVM (۰.۵۸۲ با C=4) و SVM خطی (۰.۵۸۰ با C=0.1) کمتر از یک امتیاز با آن فاصله دارند که در حد نویز validation است. complement naive Bayes حدود ۵ امتیاز عقب است، چون هر کلمه را مستقل از بقیه می‌شمارد. Ridge regression با هر دو روش تبدیل به امتیاز ضعیف است (حداکثر ۰.۴۷۵)؛ یعنی پیش‌بینی یک عدد پیوسته و بریدن آن، برای این معیار از دسته‌بندی مستقیم بدتر است. LightGBM با ۵۰۰۰ ویژگی تقریباً هم‌سطح logistic regression با همان ویژگی‌هاست (۰.۵۷۰ در برابر ۰.۵۷۲) ولی آموزشش حدود ۳۰ برابر طول کشید، و logistic regression با همه‌ی ۳۰۰ هزار ویژگی ۱.۳ امتیاز بهتر از خودش با ۵۰۰۰ ویژگی است. هر درخت فقط چند کلمه را می‌بیند، در حالی که امتیاز از جمع اثر صدها کلمه ساخته می‌شود؛ مدل خطی برای این نوع داده مناسب‌تر است.

logistic regression برای مدل TF-IDF انتخاب شد: بالاترین درستی را دارد و برای هر امتیاز احتمال می‌دهد، که برای ترکیب با ترنسفورمرها و نمایش در داشبورد لازم است.

</div>

### Training size

<div dir="rtl">

تا اینجا همه‌ی مدل‌ها روی ۱۰۰ هزار نظر آموزش دیدند. اینجا مدل منتخب روی نمونه‌های متوازن بزرگ‌تر آموزش دید. بیشترین اندازه ۵۰ هزار نظر از هر امتیاز است، چون امتیاز ۲ در داده‌ی آموزش ۵۰٬۷۲۹ نظر دارد.

</div>

In [29]:
curve = []
for per in [5000, 10000, 20000, 35000, 50000]:
    sample = fit_pool.groupby('overall').sample(per, random_state=0)
    vectorizer = word_tfidf()
    X = vectorizer.fit_transform(joined(sample))
    X_valid = vectorizer.transform(joined(valid))
    pred = lr(best_C).fit(X, sample['overall']).predict(X_valid)
    score('size', f'{5 * per:,} reviews, C={best_C}', pred)
    curve.append({'reviews': 5 * per, 'accuracy': accuracy_score(y_valid, pred)})

size_accuracy = {best_C: curve[-1]['accuracy']}
for C in [best_C / 4, best_C * 4]:
    pred = lr(C).fit(X, sample['overall']).predict(X_valid)
    score('size', f'{5 * per:,} reviews, C={C}', pred)
    size_accuracy[C] = accuracy_score(y_valid, pred)

final_C = max(size_accuracy, key=size_accuracy.get)
print('C for the final model:', final_C)

25,000 reviews, C=1                                accuracy 0.5511   macro-F1 0.5460
50,000 reviews, C=1                                accuracy 0.5674   macro-F1 0.5630
100,000 reviews, C=1                               accuracy 0.5845   macro-F1 0.5807
175,000 reviews, C=1                               accuracy 0.5993   macro-F1 0.5958
250,000 reviews, C=1                               accuracy 0.6032   macro-F1 0.6001
250,000 reviews, C=0.25                            accuracy 0.5945   macro-F1 0.5901
250,000 reviews, C=4                               accuracy 0.5898   macro-F1 0.5879
C for the final model: 1


In [30]:
fig = px.line(pd.DataFrame(curve), x='reviews', y='accuracy', markers=True,
              title='TF-IDF + logistic regression: validation accuracy vs. training size')
fig.show()

<div dir="rtl">

درستی با بیشتر شدن داده بالا می‌رود ولی رشدش کند می‌شود: ۰.۵۵۱ با ۲۵ هزار نظر، ۰.۵۸۵ با ۱۰۰ هزار و ۰.۶۰۳ با ۲۵۰ هزار. ۷۵ هزار نظر آخر فقط ۰.۴ امتیاز اضافه کرد. با ۲۵۰ هزار نظر هم C=1 بهترین ماند. نمونه‌ی متوازن بزرگ‌تر از این ممکن نیست. نمونه‌ی طبیعی بزرگ‌تر با اصلاح prior امتحان نشد، چون در اندازه‌ی برابر ۱.۲ امتیاز عقب‌تر بود.

</div>

### Final TF-IDF model

<div dir="rtl">

مدل نهایی TF-IDF: logistic regression با C=1 روی ۲۵۰ هزار نظر متوازن (۵۰ هزار از هر امتیاز)، با ویژگی‌های پایه.

</div>

In [31]:
final_sample = fit_pool.groupby('overall').sample(50000, random_state=0)
tfidf = word_tfidf()
tfidf_model = lr(final_C).fit(tfidf.fit_transform(joined(final_sample)), final_sample['overall'])

valid_pred = tfidf_model.predict(tfidf.transform(joined(valid)))
print('validation accuracy:', round(accuracy_score(y_valid, valid_pred), 4))
print('within one star:', round((np.abs(valid_pred - y_valid) <= 1).mean(), 4))

validation accuracy: 0.6032
within one star: 0.9217


In [32]:
cm = confusion_matrix(y_valid, valid_pred, normalize='true')
fig = px.imshow(cm, x=['1', '2', '3', '4', '5'], y=['1', '2', '3', '4', '5'], text_auto='.2f', color_continuous_scale='Blues',
                labels=dict(x='predicted rating', y='true rating', color='share'),
                title='TF-IDF model on validation (each row sums to 1)')
fig.show()

<div dir="rtl">

وزن‌های مدل نشان می‌دهند کدام کلمه‌ها بیشترین اثر را به سمت هر امتیاز دارند:

</div>

In [33]:
words = tfidf.get_feature_names_out()
pd.DataFrame({f'rating {rating}': words[np.argsort(coef)[::-1][:15]]
              for rating, coef in zip(tfidf_model.classes_, tfidf_model.coef_)})

,rating 1,rating 2,rating 3,rating 4,rating 5
0,junk,2 stars,3 stars,4 stars,great
1,useless,two stars,three stars,good,excellent
2,terrible,not,ok,four stars,perfect
3,garbage,2 star,okay,great,awesome
4,not,poor,3 star,nice,amazing
5,horrible,disappointed,but,star,love
6,1 star,disappointing,stars,a star,perfectly
7,one star,not worth,however,my only,best
8,worst,to 2,to 3,overall,fantastic
9,don't buy,returned,decent,to 4,highly


<div dir="rtl">

درستی مدل روی validation ۰.۶۰۳ است و ۹۲٪ پیش‌بینی‌ها حداکثر یک ستاره با امتیاز واقعی فاصله دارند. امتیازهای ۱ و ۵ بهتر تشخیص داده می‌شوند (۰.۷۰ و ۰.۷۹) و امتیاز ۲ سخت‌ترین است (۰.۴۶). بیشتر خطاها به امتیاز کناری می‌روند.

کلمه‌های مؤثر معنی‌دارند: برای ۱ junk، useless و don't buy و برای ۵ great، excellent و perfect. خیلی از نویسنده‌ها امتیاز را در متن هم می‌نویسند (2 stars، three stars). عبارت‌هایی مثل 2 stars فقط به این دلیل در مدل هستند که الگوی token عددها را نگه می‌دارد. برای ۳ کلمه‌هایی مثل ok، okay، but و however آمده و برای ۴ عبارت‌هایی مثل my only و the only (احتمالاً از جمله‌هایی مثل my only complaint) و one star (کم کردن یک ستاره). not در ستون‌های ۱ و ۲ است؛ حذف آن به‌عنوان stop word اطلاعات مهمی را دور می‌ریخت.

</div>

<div dir="rtl">

مدل و احتمال‌هایش برای validation، holdout و test ذخیره می‌شوند. نوت‌بوک آخر این احتمال‌ها را با ترنسفورمرها مقایسه و ترکیب می‌کند، و داشبورد از خود مدل استفاده می‌کند. درستی روی holdout اینجا حساب نمی‌شود.

</div>

In [34]:
import joblib

joblib.dump({'vectorizer': tfidf, 'model': tfidf_model}, path + 'part3_tfidf.joblib')

probs = []
for name, df in [('valid', valid), ('holdout', train[train['set'] == 'holdout']), ('test', test)]:
    p = tfidf_model.predict_proba(tfidf.transform(joined(df)))
    probs.append(pd.DataFrame(p, columns=['p1', 'p2', 'p3', 'p4', 'p5'], index=df.index).assign(set=name))
probs = pd.concat(probs)
probs.to_parquet(path + 'part3_tfidf_probs.parquet')
probs['set'].value_counts()

,count
set,
test,20000
valid,10000
holdout,10000


## Summary

<div dir="rtl">

- بعد از حذف نظرهای تکراری و نسخه‌های دیگرِ یک متن، ۸۰۷٬۴۱۳ نظر ماند. متن ۶.۳٪ نظرهای test در train هم هست، ولی امتیازشان از train کپی نشد.
- test به احتمال زیاد از هر امتیاز سهم برابری دارد (برآورد هر امتیاز بین ۱۶٪ و ۲۳٪)، در حالی که در train امتیاز ۵ حدود ۵۵٪ است. پس validation و holdout متوازن‌اند (۲۰۰۰ نظر از هر امتیاز) و هیچ متن یا جفت نویسنده و محصولی از آن‌ها در داده‌ی آموزش نیست.
- آموزش روی نمونه‌ی متوازن از اصلاح prior (۱.۲ امتیاز) و class_weight (۲ امتیاز) بهتر است. مدل بدون اصلاح ۹ امتیاز عقب است.
- summary ۲.۴ امتیاز، جفت‌کلمه‌ها ۳.۷ امتیاز و نگه داشتن stop wordها ۴.۵ امتیاز اثر دارند. stemming، n-gramهای حرفی و ویژگی‌های جدا برای summary کمکی نکردند.
- logistic regression، NB-SVM و SVM خطی هم‌سطح‌اند. naive Bayes، Ridge و LightGBM عقب‌ترند و LightGBM حدود ۳۰ برابر کندتر است.
- مدل TF-IDF نهایی (logistic regression روی ۲۵۰ هزار نظر متوازن) روی validation درستی ۰.۶۰۳ دارد و ۹۲٪ پیش‌بینی‌هایش حداکثر یک ستاره خطا دارند. امتیاز ۲ سخت‌ترین (۰.۴۶) و امتیاز ۵ آسان‌ترین (۰.۷۹) است.

</div>